# cAPTure: XGB-P sanity controls

This CPU-only notebook audits the completed depth-5 XGB-P development run. It does not alter the primary models or OOF scores, access test scenarios, select thresholds, or choose new model features. The shuffled-label negative control uses a deterministic one-in-100 packet sample to keep the check practical on Colab CPU; it is not a replacement for full-data validation.

## 1. Mount Drive and load the repository

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/utils/capture_xgb_p_sanity.py", "code/python/tests/test_capture_xgb_p_sanity.py", "code/python/requirements-capture-xgb.txt"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("XGB-P sanity environment is ready.")
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

Mounted at /content/drive
XGB-P sanity environment is ready.
Repository commit: 182c59adce7da1ceec88a4029147cf801cd841d3


## 2. Run synthetic checks

In [3]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for pattern in ("test_capture_xgb_p.py", "test_capture_xgb_p_sanity.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "code/python/tests"), "-p", pattern, "-v"], env=test_environment, cwd=PROJECT_ROOT, check=True)
print("Synthetic XGB-P checks passed.")

Synthetic XGB-P checks passed.


## 3. Bind the completed development artifacts

Set `RUN_ID` to an existing sanity run ID when resuming. The baseline run is read-only. Its archived manifest predates the newly frozen context-feature proposal, so the sanity runner checks the baseline configuration, preprocessing hash, scenario assignments, and prepared packet hashes rather than requiring identical manifest-file hashes.

In [4]:
import pandas as pd
from IPython.display import display
from utils.capture_data import load_manifest, sha256_file
from utils.capture_xgb_p_sanity import (
    run_shuffled_label_control,
    run_single_feature_diagnostics,
    summarize_shuffled_label_controls,
    validate_shuffled_label_control,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
RUN_ID = None  # Replace with an earlier sanity run ID only when resuming.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p_sanity"
SANITY_RUN_DIR = DRIVE_ROOT / "xgb_p_sanity_runs" / RUN_ID
BATCH_SIZE = 100_000
CPU_THREADS = 2
for path in (PREPARED_RUN_DIR, PREPROCESSING_AUDIT_DIR, BASELINE_RUN_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required Drive run is missing: {path}")
manifest = load_manifest(MANIFEST_PATH)
print("Baseline run:", BASELINE_RUN_DIR)
print("Sanity output:", SANITY_RUN_DIR)
print("Negative-control sample modulus:", manifest["training"]["xgb_p_sanity_controls"]["shuffled_training_labels"]["sample_modulus"] )

Baseline run: /content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p
Sanity output: /content/drive/MyDrive/capture_gate0/xgb_p_sanity_runs/20260919T184528_784079Z_xgb_p_sanity
Negative-control sample modulus: 100


## 4. Inspect fixed single-feature diagnostics

These are descriptive held-out ROC-AUC values for declared individual packet features. The best-direction value uses the validation labels to describe scalar ranking separation and must not be used to select features or tune the model. A nonlinear one-feature tree could behave differently.

In [5]:
feature_report_path = SANITY_RUN_DIR / "single_feature_diagnostics.json"
if feature_report_path.exists():
    feature_report = json.loads(feature_report_path.read_text(encoding="utf-8"))
    if feature_report.get("status") != "single_feature_oof_diagnostics_complete" or feature_report.get("manifest_sha256") != sha256_file(MANIFEST_PATH) or feature_report.get("code_sha256") != sha256_file(PROJECT_ROOT / "code/python/utils/capture_xgb_p_sanity.py"):
        raise ValueError("Existing single-feature diagnostics do not match the current manifest.")
else:
    feature_report = run_single_feature_diagnostics(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, baseline_run_dir=BASELINE_RUN_DIR, output_path=feature_report_path, batch_size=BATCH_SIZE)
feature_table = pd.DataFrame(feature_report["rows"])
display(feature_table.sort_values(["scenario", "best_direction_roc_auc_diagnostic"], ascending=[True, False]))
print("Feature diagnostics saved:", feature_report_path)

,fold,scenario,feature,packets,distinct_values,raw_direction_roc_auc,best_direction_roc_auc_diagnostic,baseline_packet_roc_auc
0,A,train_dollar_char,is_mqtt,2882555,2,0.195023,0.804977,0.918163
2,A,train_dollar_char,frame_length,2882555,397,0.322077,0.677923,0.918163
1,A,train_dollar_char,tcp_destination_port_role_mqtt_messaging,2882555,2,0.328259,0.671741,0.918163
5,A,train_dollar_char,tcp_flag_ack,2882555,2,0.412560,0.587440,0.918163
6,A,train_dollar_char,mqtt_qos_0,2882555,2,0.576187,0.576187,0.918163
3,A,train_dollar_char,is_arp,2882555,2,0.508100,0.508100,0.918163
4,A,train_dollar_char,destination_is_multicast,2882555,2,0.497419,0.502581,0.918163
21,B,train_empty_conn,is_mqtt,1175779,2,0.185025,0.814975,0.990654
23,B,train_empty_conn,frame_length,1175779,245,0.224146,0.775854,0.990654
22,B,train_empty_conn,tcp_destination_port_role_mqtt_messaging,1175779,2,0.229278,0.770722,0.990654


Feature diagnostics saved: /content/drive/MyDrive/capture_gate0/xgb_p_sanity_runs/20260919T184528_784079Z_xgb_p_sanity/single_feature_diagnostics.json


## 5. Run the sampled shuffled-label negative control

Each fold trains a separate depth-5, 200-round XGBoost model on its sampled training scenarios after a within-scenario label permutation. Validation uses the original labels and the same deterministic row sample. A complete existing fold is verified; incomplete outputs are not overwritten.

In [6]:
def run_or_verify_control(fold):
    output_dir = SANITY_RUN_DIR / "shuffled_labels" / f"fold_{fold}"
    if output_dir.exists():
        print(f"Verifying existing shuffled-label fold {fold}...")
        report = validate_shuffled_label_control(output_dir, fold)
        if report["manifest_sha256"] != sha256_file(MANIFEST_PATH) or report["code_sha256"] != sha256_file(PROJECT_ROOT / "code/python/utils/capture_xgb_p_sanity.py"):
            raise ValueError("Existing control was produced by a different manifest or trainer.")
        return report
    print(f"Training sampled shuffled-label fold {fold}...")
    return run_shuffled_label_control(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR, baseline_run_dir=BASELINE_RUN_DIR, output_dir=output_dir, fold=fold, batch_size=BATCH_SIZE, nthread=CPU_THREADS)

control_a = run_or_verify_control("A")
display(pd.DataFrame.from_dict(control_a["validation"], orient="index")[["sampled_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

Training sampled shuffled-label fold A...


,sampled_packets,packet_roc_auc,packet_pr_auc_diagnostic
train_dollar_char,28826,0.538779,0.510411
train_slash_char,64256,0.576159,0.800442
train_sub_exf,22259,0.519327,0.289639


In [7]:
control_b = run_or_verify_control("B")
display(pd.DataFrame.from_dict(control_b["validation"], orient="index")[["sampled_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

Training sampled shuffled-label fold B...


,sampled_packets,packet_roc_auc,packet_pr_auc_diagnostic
train_empty_conn,11758,0.405510,0.313308
train_qos_mid,14998,0.504311,0.485709


## 6. Review the negative-control summary

A near-0.5 macro ROC-AUC is reassuring but does not prove the absence of leakage. A value above the predeclared 0.6 review line requires investigation before proceeding to XGB-P+T. The univariate diagnostics may reveal strong protocol fingerprints even if the label-shuffle control passes.

In [8]:
control_summary = summarize_shuffled_label_controls(SANITY_RUN_DIR / "shuffled_labels")
display(pd.DataFrame.from_dict(control_summary["scenario_metrics"], orient="index")[["fold", "sampled_packets", "packet_roc_auc"]])
print("Shuffled-label hierarchical macro ROC-AUC:", control_summary["hierarchical_macro_oof_packet_roc_auc"])
print("Investigation required:", control_summary["review_required"])
print("This is a sampled negative control, not a full-data model estimate.")

,fold,sampled_packets,packet_roc_auc
train_dollar_char,A,28826,0.538779
train_slash_char,A,64256,0.576159
train_sub_exf,A,22259,0.519327
train_empty_conn,B,11758,0.405510
train_qos_mid,B,14998,0.504311


Shuffled-label hierarchical macro ROC-AUC: 0.4998326345817774
Investigation required: False
This is a sampled negative control, not a full-data model estimate.


## 7. Investigate the MQTT protocol shortcut

  Measure the existing OOF model within MQTT and non-MQTT packets, then compare protocol prevalence and packet lengths by attack step. These diagnostics do not change the primary model or select features.



In [9]:
import gc
import numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

from utils.capture_prepare import load_packet_schema
from utils.capture_xgb_p import validate_xgb_p_fold_run

packet_artifact = load_packet_schema(PACKET_SCHEMA_PATH)["output_artifact"]
protocol_rows = []

for fold, split in manifest["validation"]["folds"].items():
    fold_dir = BASELINE_RUN_DIR / "depth5_primary" / f"fold_{fold}"
    baseline_report = validate_xgb_p_fold_run(
        fold_dir, fold, "depth5_primary"
    )

    for scenario in split["validate"]:
        prepared_path = PREPARED_RUN_DIR / scenario / packet_artifact
        oof_path = fold_dir / baseline_report["validation"][scenario]["oof_artifact"]

        prepared = pq.read_table(
            prepared_path,
            columns=["source_row_id", "binary_label", "is_mqtt"],
        )
        oof = pq.read_table(
            oof_path,
            columns=["source_row_id", "binary_label", "score"],
        )

        prepared_ids = prepared["source_row_id"].to_numpy(zero_copy_only=False)
        oof_ids = oof["source_row_id"].to_numpy(zero_copy_only=False)
        labels = oof["binary_label"].to_numpy(zero_copy_only=False)
        prepared_labels = prepared["binary_label"].to_numpy(zero_copy_only=False)
        mqtt = prepared["is_mqtt"].to_numpy(zero_copy_only=False)
        scores = oof["score"].to_numpy(zero_copy_only=False)

        if not np.array_equal(prepared_ids, oof_ids):
            raise ValueError(f"Packet alignment changed for {scenario}.")
        if not np.array_equal(prepared_labels, labels):
            raise ValueError(f"Packet labels changed for {scenario}.")
        if not np.isin(mqtt, [0, 1]).all():
            raise ValueError(f"Invalid MQTT indicator in {scenario}.")

        for mqtt_value, group_name in ((0, "non_mqtt"), (1, "mqtt")):
            selected = mqtt == mqtt_value
            group_labels = labels[selected]
            group_scores = scores[selected]

            protocol_rows.append({
                "fold": fold,
                "scenario": scenario,
                "protocol_group": group_name,
                "packets": int(selected.sum()),
                "normal_packets": int((group_labels == 0).sum()),
                "attack_packets": int((group_labels == 1).sum()),
                "packet_roc_auc": (
                    float(roc_auc_score(group_labels, group_scores))
                    if np.unique(group_labels).size == 2
                    else np.nan
                ),
            })

        del prepared, oof, prepared_ids, oof_ids
        del labels, prepared_labels, mqtt, scores
        gc.collect()

protocol_table = pd.DataFrame(protocol_rows)
display(protocol_table)


,fold,scenario,protocol_group,packets,normal_packets,attack_packets,packet_roc_auc
0,A,train_dollar_char,non_mqtt,1392721,398942,993779,0.954614
1,A,train_dollar_char,mqtt,1489834,1306061,183773,0.685232
2,A,train_slash_char,non_mqtt,5103958,398942,4705016,0.966539
3,A,train_slash_char,mqtt,1321558,1306061,15497,0.929897
4,A,train_sub_exf,non_mqtt,917412,398942,518470,0.940266
5,A,train_sub_exf,mqtt,1308395,1306061,2334,0.951502
6,B,train_empty_conn,non_mqtt,692973,268508,424465,0.977302
7,B,train_empty_conn,mqtt,482806,478298,4508,0.976437
8,B,train_qos_mid,non_mqtt,962659,268508,694151,0.958787
9,B,train_qos_mid,mqtt,537058,478298,58760,0.718113


In [10]:
import duckdb

import duckdb

profile_rows = []
connection = duckdb.connect()

try:
    for scenario in manifest["gate0"]["modes"]["FULL_DEV"]:
        prepared_path = PREPARED_RUN_DIR / scenario / packet_artifact

        profile = connection.execute(
            """
            SELECT
                CASE
                    WHEN binary_label = 0 THEN 'normal'
                    ELSE attack_step
                END AS traffic_group,
                binary_label,
                COUNT(*) AS packets,
                SUM(CASE WHEN is_mqtt = 1 THEN 1 ELSE 0 END) AS mqtt_packets,
                AVG(CAST(is_mqtt AS DOUBLE)) AS mqtt_fraction,
                approx_quantile(CAST(frame_length AS DOUBLE), 0.05) AS length_q05,
                approx_quantile(CAST(frame_length AS DOUBLE), 0.50) AS length_median,
                approx_quantile(CAST(frame_length AS DOUBLE), 0.95) AS length_q95
            FROM read_parquet(?)
            GROUP BY 1, 2
            ORDER BY binary_label, packets DESC
            """,
            [str(prepared_path)],
        ).fetchdf()

        profile.insert(0, "scenario", scenario)
        profile_rows.append(profile)
finally:
    connection.close()

protocol_profile = pd.concat(profile_rows, ignore_index=True)

class_profile = protocol_profile.groupby(
    ["scenario", "binary_label"], as_index=False
)[["packets", "mqtt_packets"]].sum()
class_profile["mqtt_fraction"] = (
    class_profile["mqtt_packets"] / class_profile["packets"]
)

print("MQTT prevalence by scenario and class")
display(class_profile)

print("MQTT prevalence and packet length by attack step")
display(protocol_profile)



MQTT prevalence by scenario and class


,scenario,binary_label,packets,mqtt_packets,mqtt_fraction
0,train_dollar_char,0,1705003,1306061.0,0.766017
1,train_dollar_char,1,1177552,183773.0,0.156064
2,train_empty_conn,0,746806,478298.0,0.640458
3,train_empty_conn,1,428973,4508.0,0.010509
4,train_qos_mid,0,746806,478298.0,0.640458
5,train_qos_mid,1,752911,58760.0,0.078044
6,train_slash_char,0,1705003,1306061.0,0.766017
7,train_slash_char,1,4720513,15497.0,0.003283
8,train_sub_exf,0,1705003,1306061.0,0.766017
9,train_sub_exf,1,520804,2334.0,0.004482


MQTT prevalence and packet length by attack step


,scenario,traffic_group,binary_label,packets,mqtt_packets,mqtt_fraction,length_q05,length_median,length_q95
0,train_empty_conn,normal,0,746806,478298.0,0.640458,70.000000,74.000000,164.134988
1,train_empty_conn,nmap_10_T4,1,255667,48.0,0.000188,64.000000,64.000000,64.000000
2,train_empty_conn,brute_force_timing,1,107515,216.0,0.002009,70.000000,189.317741,1518.000000
3,train_empty_conn,empty_conn_ddos,1,28269,2925.0,0.103470,64.000000,71.900197,1514.000000
4,train_empty_conn,nmap_mqtt,1,14168,482.0,0.034020,64.000000,64.000000,600.089821
5,train_empty_conn,nmap_banner,1,12649,134.0,0.010594,64.000000,64.000000,130.389006
6,train_empty_conn,empty_conn,1,6650,651.0,0.097895,64.000000,71.995137,1514.000000
7,train_empty_conn,nmap_sub,1,2277,30.0,0.013175,64.000000,64.000000,132.310784
8,train_empty_conn,mqtt_cat,1,1171,18.0,0.015371,70.000000,118.000000,474.570790
9,train_empty_conn,sftp_inst,1,607,4.0,0.006590,70.000000,150.000000,1518.000000
